Based on "warp_supernova_select" mixed up with warp_template

- Look at things for each narrow class - will be combined in a following step. But - do we not need to know which class we are working towards in order to find which templates should be considered true? 
- Define subset of BTS supernovae below some redshift cat, with type and minimal data quality.
- For each SN and template compbo, find the warp coefficients that best match. Store these if "successful"

Taking over from v2_I, where we assume that e.g. redshift limits has already been set and classes are used as in v2_0.

The goal is to parse the btsfits summary files and collect the results for whatever combination is needed for your warp template classes.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re, sys
import pickle
import json
from astropy.cosmology import Planck13 as cosmo
import pymongo
import sncosmo
sys.path.append('/Users/jnordin/github/ampelFeb25')
from warpTemplate import WarpfitTemplateLoader
from warpTemplate import get_template_correction, get_warpedTimeSeriesModel,  add_warpclasses, TEMPLATE_CLOSE_TYPES
#from warptemplate import get_ztftable_from_ampel, get_warpedTimeSeriesModel, get_template_correction
from scipy.stats.distributions import chi2

In [ ]:
# Load BTS information - mainly used to collect correct BTS name classes 
df_bts = pd.read_csv('/Users/jnordin/data/ztf/bts/bts_explorer_241122.csv')
df_bts = add_warpclasses(df_bts, purge=True)

In [ ]:
# Which warp class to parse for?
# Category: (n)arrow, (e)xtended, (w)ide or (a)ll?
category = 'a'
# Id for class to run for
cid = 1

In [ ]:
classlist = list(set(df_bts['type_'+category]))
print('Target class list {} from category {}'.format(classlist, 'type_'+category))

In [ ]:
class_name = classlist[cid]
#class_name = 'SN Ibc (e)'
print('Target class {} from category {}'.format(class_name, 'type_'+category))

In [ ]:
# Get the list of narrow classes (for which there are btsfits files) for this subset
tmask = (df_bts['type_'+category]==class_name)

In [ ]:
process_classes = list( set( df_bts.loc[tmask]['type_n'] ) )

In [ ]:
print('Will combine files for narrow classes: ', process_classes)

In [ ]:
close_templates = [key for key, val in TEMPLATE_CLOSE_TYPES.items() if class_name in val]

In [ ]:
print('these are the template classes which will be called close:', close_templates)

In [ ]:
def get_ztftable_from_ampel(ztfid: str, dbhandle, include_sigma: float = 5., **kwarg):
    """
    Given a ZTF name and a local AMPEL DB:
    - Retrieve available photometric data.
    - Reject outliers. 
    - Return an astropy table useful e.g. for sncosmo. 

    Parameters:
    - ztfid: str : ZTF name, e.g. ZTF18aaayemw
    - dbhandle: Database : AMPEL MongoDB handle
    - inclusion_sigma: float : Sigma threshold for outlier rejection.
    - kwarg: dict : Additional arguments added to table meta.

    tab = get_ztftable_from_ampel('ZTF22aaa', dbhandle, inclusion_sigma=5., z=0.03, type='SN Ia')

    """

    # Load photopoints from AMPEL DB
    tabulators = [
        ZTFFPTabulator(inclusion_sigma=include_sigma)
        ]
    tab = get_db_table(ztfid, database=dbhandle, tabulators=tabulators)
    tab.sort('time')

    tab.meta = {
            'object_id':ztfid,
            **kwarg
        }

    return tab


def get_db_table( name, database, tabulators ):
    """
    For ZTF name, get photopoints and then tables. 
    """

    # Name
    if isinstance(name, int):    
        # Assuming this is already a DB stock
        stock = int
    elif re.search('ZTF', name):
        stock = ZTFIdMapper.to_ampel_id(name)
    else:
        raise ValueError(f"Cannot parse {name}" )

    # Obtain photopoints
    dps = [dp for dp in database.t0.find({'stock':stock})]

    # Convert to table(s)
    ftables = [
        tabulator.get_flux_table(dps) for tabulator in tabulators
    ]
    if len(ftables)>1:
        raise NotImplementedError("Debug appending two tabulator tables.")
    return ftables.pop(0)

def apply_floor_and_normalize(p, floor):
    p = np.array(p, dtype=float)
    n = len(p)
    
    if floor * n > 1:
        raise ValueError("Floor too large to maintain sum=1")
    
    # Start by assigning the floor everywhere
    result = np.full(n, floor)
    
    # Remaining probability mass to distribute
    remaining = 1 - floor * n
    
    # Original excess above floor
    excess = np.maximum(p - floor, 0)
    
    if excess.sum() > 0:
        result += remaining * (excess / excess.sum())
    else:
        # If all values were below floor, distribute evenly
        result += remaining / n
    
    return result

In [ ]:
import warnings
from iminuit.util import IMinuitWarning

warnings.filterwarnings("ignore", category=IMinuitWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

In [ ]:
client = pymongo.MongoClient()
db = client.bts_ipacfp_strictbase    # Only final lc. try bts_ipacfp_strictbase_full for all alerts

In [ ]:
# So, what are our recommendeded limits?
# These are the things we can try
# Ok, fix these ... seems like we cannot really increase the requirements 
# and keep sufficient data. Instead, think of a ranking of data
# and use the best fits.
min_bands= 2
min_peakndof = 3
require_peak_good = False
chidofmax = 6

# Fixed params
fdir = '/Users/jnordin/data/models/sncosmo/'
peakfit = True          # Only use data round peak to estimate fit quality 

from ampel.ztf.util.ZTFIdMapper import ZTFIdMapper
from ampel.ztf.view.ZTFFPTabulator import ZTFFPTabulator

# Corresponding values 
errfloor = 0.0
flux_frac_disperson = 0.02   # Here we keep only a base calibration uncertainty - the variability is what we are fitting



### Start processing

In [ ]:
def combine_typefits( dictlI, dictlII ):
    """
    Combine the internal dicts of two lists of dicts.
    """
    for templatename, datalist in dictlII.items():
        dictlI[templatename].extend( datalist )
    return dictlI 

In [ ]:
typefitdata = None
for readclass in process_classes:
    if readclass=='SN Ia':
        print('... skipping normal SNe Ia - did not run the bts sample.')
        continue
    rootname = 'btsfitsv2_{}.json'.format(readclass.replace('/',''))
    fname = fdir+rootname
    # Open data
    with open(fname, "r") as infile:
        if typefitdata is None:
            typefitdata = json.loads(infile.read())
        else:
            typefitdata = combine_typefits( 
                typefitdata, json.loads(infile.read()) 
            )
    
    print(readclass, len(typefitdata))

In [ ]:
def get_salt_cosmofit( fitlist,
                        chidofmax=3,    # Max chi/dof to be counted as good fit
                        truetypes = [], peakfit=True,
                     ):
    """
    Go through the input list of saltfits. Return a list of the subset which fulfills
    selection criteria as specified. The idea is that the provided list of templates
    are "good enough" for template construction.

    Parameters:
    - 
    determine whether the fit is good and whether it 
    could be considered for cosmology.

    If the model class included in truetypes, classification set to correct.

    Nothing returned for failed fits.

    Return a pandas df
    """

    outd = []

    for snfit in fitlist:
        if not snfit['success']:
            continue
        snd = {
            k:snfit[k] for k in ['z', 'chisq', 'ndof', 'peakmag', 'chidof', 'id', 'nbr_bands', 'ndet','class',
                                 'peakchi','peakdet','earlydet','peakbands','peak_good', 
                                 'presum', 'postsum',  'earlydet', 'thendet', 'postdet',
                                ]
        }
        fitp = snfit['ndet']-snfit['ndof']
        snd['aic'] = 2 * fitp + snd['chisq'] # 4 dof, ln L = - 0.5 chi 
        snd['peakndof'] = snfit['peakdet']-fitp

        if peakfit:
            if not snd['peakndof']>0:
                continue
            snd['peakchisqdof'] = snfit['peakchi']/snd['peakndof']
            snd['peakaic'] = 2 * fitp + snfit['peakchi'] # 4 dof, ln L = - 0.5 chi
            snd['chidof'] = snfit['peakchi']/snd['peakndof']
        else:
            snd['chidof'] = snfit['chidof']

            
        # Check for good, correct fit
        snd['correct'] = False
        if snd['class'] in truetypes:
            snd['correct'] = True
        if (snd['chidof'] < chidofmax):
            snd['goodfit'] = True
        else:
            snd['goodfit'] = False

        outd.append(snd)

    return pd.DataFrame.from_dict(outd)


def get_timeseries_goodfit( fitlist,
                        chidofmax=3,    # Max chi/dof to be counted as good fit
                        truetypes = [],
                        peakfit=True,
                          ):
    """
    Go through the input list of an sncosmo timeseries model with added dust. 
    For each, determine whether the fit is good.

    Nothing returned for failed fits.

    Assumes fit made with amplitude, rv and ebv. Should probably redo without fitting both ... 

    Return a pandas df
    """

    outd = []

    for snfit in fitlist:
        if not snfit['success']:
            continue
        snd = {
            k:snfit[k] for k in ['z', 'chisq', 'ndof', 'peakmag', 'chidof', 'id', 'nbr_bands', 'ndet','class',
                                 'peakchi','peakdet','earlydet','peakbands','peak_good',
                                 'presum', 'postsum',  'earlydet', 'thendet', 'postdet',                                
                                ]
                                 
        }
        fitp = snfit['ndet']-snfit['ndof']
        snd['aic'] = 2 * fitp + snd['chisq'] # ln L = - 0.5 chi 

        snd['peakndof'] = snfit['peakdet']-fitp

        if peakfit:
            if not snd['peakndof']>0:
                continue
            snd['peakchisqdof'] = snfit['peakchi']/snd['peakndof']
            snd['peakaic'] = 2 * fitp + snfit['peakchi'] # 4 dof, ln L = - 0.5 chi
            snd['chidof'] = snfit['peakchi']/snd['peakndof']
        else:
            snd['chidof'] = snfit['chidof']


        if (snd['chidof'] < chidofmax):
            snd['goodfit'] = True
        else:
            snd['goodfit'] = False

            
        if snd['class'] in truetypes:
            snd['correct'] = True
        else:
            snd['correct'] = False

        outd.append(snd)

    return pd.DataFrame.from_dict(outd)


In [ ]:
modelfits = {}
for modelname, modeldata in typefitdata.items():
    if re.search('salt', modelname):
        df = get_salt_cosmofit( modeldata, truetypes=close_templates, peakfit=peakfit, chidofmax=chidofmax )
    else:
        df = get_timeseries_goodfit( modeldata, truetypes=close_templates, peakfit=peakfit, chidofmax=chidofmax )
    df.insert(0,'model', modelname) 
    
    modelfits[modelname] = df

In [ ]:
# Combine to one df
dfall = pd.concat( modelfits.values(), ignore_index=True )

In [ ]:
print('Total transient/model pairs {} of correct type {} with good fit {} both: {}'.format(
    dfall.shape[0], 
    sum(dfall['correct']==True), 
    sum(dfall['goodfit']==True), 
    sum( (dfall['correct']==True) & (dfall['goodfit']==True) )
) )

In [ ]:
# Limiting to good fits
dfall = dfall.loc[dfall['goodfit']]
print(dfall.shape[0])

In [ ]:
print('Total transient/model pairs {} of correct type {}'.format(
    dfall.shape[0], 
    sum(dfall['correct']==True), 
) )

In [ ]:
# Ok, so this is where we skip directly to the warp stage. We iterate through each row:
# - Fit a warp and plot a warp template, looking at the parameer for this.
# - Possibly we do some quality cut arleady now and we do not include template of different classes.
# - as a start, only run for the target objects from Suleyman ...

In [ ]:
# Selection params
# These are the things we need to investigate
template_count = 5 # We will try to include up to this many templates
min_draw_prob = 0.01   # Min template prob *after* fit
good_warpfit_sf = 0.5

In [ ]:
len(set(dfall['id']))

In [ ]:
chicomp = {'prechi':[],'postsf':[]}
full_warplist = {}
fitprop = ['t0', 'amplitude', 'hostebv']
i=0
for id_value, group in dfall.groupby('id'):
    print(i,id_value)
    i+=1
    #if not id_value=="ZTF20abaszgh":
    #    continue

    ordered = (
        group.assign(
            priority=(
                ((group['correct']) & (group['goodfit'])) * 2 +
                ((group['correct']) & (~group['goodfit'])) * 1
            )
        )
#        .query('priority > 0')
        .sort_values(['priority', 'chidof'], ascending=[False, True])
    )
    #use = ordered.head( template_count )
    #print(id_value, len(use))
    #continue

    sn_warplist = []
    goodfits = 0
    for row in ordered.itertuples():
        row = row._asdict()
        row['class'] = row['_10']
        
        # Gradually try to add more warptemplates
        # saving if successful
        # abort iterator after reaching limit

        # Get SN table
        # Lets grab some data, starting with the target SN
        tab = get_ztftable_from_ampel( row['id'], db, 
                                  redshift=float(row['z']), 
                                  include_sigma=5,
                                  type = row['class'] )
        # Add errorfloor
        tab['fluxerr'] = np.sqrt((errfloor*np.mean(tab['flux']))**2+(tab['fluxerr'])**2)
        tab['fluxerr'] = np.sqrt((flux_frac_disperson*tab['flux'])**2+(tab['fluxerr'])**2)   

    
        # Get correction factors
        if re.search('salt', row['model']):
            continue
        mdict = get_template_correction( tab, row['model'], z=float(row['z']), 
                                        pull_cut=10, plot_dir=None, 
                                        spline_lam=0.001, require_phasecoverage=False)

        if not mdict['success']:
            continue


        wm = get_warpedTimeSeriesModel( 
            name=row['id']+'_'+row['model'], original_template_name=row['model'], warpdata=mdict, 
            z=float(row['z']), original_template_version=None 
        )

        try:
            wresult, wfitted_model = sncosmo.fit_lc( tab, wm, fitprop, )
        except RuntimeError:
            continue

        # We need some way of verifying that the template peak fit time is not before or after the data
        if (tab['time'].min()>wresult['parameters'][1] or tab['time'].max()<wresult['parameters'][1]):
            #print('peak outside data range - skippin')
            continue
        if ( (tab['time'].min()-wresult['parameters'][1])<(wm.source.minphase()-3) ):
            #print('first detection prior to template start - we do not allow this')
            continue
        # We do not want cases where the first detection is more than ten days after peak
        if ( (tab['time'].min()-wresult['parameters'][1])>5 ):
            #print('first detection prior more than five days after peak - we do not allow this')
            continue


        # Fit acceptable - look at surviabibility of the warped model
        row['sf'] =  chi2.sf(wresult['chisq'],wresult['ndof'])
        chicomp['prechi'].append( row['chidof'] ) 
        chicomp['postsf'].append( row['sf'] ) 

#        print(row)
#        print(row['sf'])
    
        # Fit done - start saving
        row['mdict'] = mdict
        row['wresult'] = wresult

        if row['sf'] < min_draw_prob:
            # Fitted model not looking very good
            #print('low prob fitted model, skipping')
            continue
        if row['sf']>good_warpfit_sf:
            goodfits+=1

        # Evaluate whether this template should be considered "gold"
        if (row['sf']>good_warpfit_sf) and row['priority']==2:
            row['quality']='gold'
        elif (row['sf']>good_warpfit_sf) and row['priority']==1:
            row['quality']='silver'
        else:
            row['quality']='bronze'
        
        sn_warplist.append( row )

        # Q: So what probability should we use to select templates?
        # the raw probability from fit without warping, or the one with warping?
        # If we use the latter we need to iterate through more to compare .... 
        # Maybe start with checkig the correlation between probabilities?
        # 


        
        if goodfits>=template_count:
            print('... reached target!')
            break

    if len(sn_warplist)==0:
        print('No template fits for', id_value)
        continue
    
    print(len(sn_warplist), len(ordered))
    # Analyse results for this sn
    # df_subset.loc[:,'draw_prob'] 
    drawprobs = apply_floor_and_normalize( 
        [modfit['sf'] for modfit in sn_warplist], min_draw_prob 
    )
    for k, prob in enumerate(drawprobs):
        sn_warplist[k]['draw_prob'] = prob 

    full_warplist[id_value] = sn_warplist

    #if len(full_warplist)>10:
    #    break


In [ ]:
plt.plot(chicomp['prechi'], chicomp['postsf'], 'o')

In [ ]:
accounting = []
for sn, wdata in full_warplist.items():
    account = 0
    for mdict in wdata:
        if mdict['quality']=='gold':
            account+=3
        elif mdict['quality']=='silver':
            account+=2
        elif mdict['quality']=='bronze':
            account+=1
    accounting.append(account)

In [ ]:
plt.hist( accounting )

In [ ]:
tocut = ['Index', 'chisq', 'ndof', 
         '_10', 'earlydet', 'peakbands', 'presum', 'postsum', 'thendet', 'postdet', 'aic', 
         'peakchisqdof', 'peakaic', 'goodfit', 'wresult']


In [ ]:
mdict_tocut = [
    'dps_init', 't0', 'amplitude', 'hostebv', 'hostr_v', 'dps_fcut', 'dps_tcut', 
    'dps_allcut', 'success', 'chisq', 'ndof', 'errors', 'chidof', 'absmag', 'corrdata'
]

In [ ]:
def remove_keys(obj, keys_to_remove):
    if isinstance(obj, dict):
        return {
            k: remove_keys(v, keys_to_remove)
            for k, v in obj.items()
            if k not in keys_to_remove
        }
    elif isinstance(obj, list):
        return [remove_keys(item, keys_to_remove) for item in obj]
    else:
        return obj

In [ ]:
full_warplist_II = remove_keys(full_warplist, tocut)

In [ ]:
full_warplist_III = remove_keys(full_warplist_II, mdict_tocut)

In [ ]:
storefile = '/Users/jnordin/data/models/sncosmo/warpmod/warpcoeffs_v2_'+re.sub(r'/', '', class_name)+'.pkl'

In [ ]:
with open(storefile, 'wb') as file:
    pickle.dump(full_warplist_III, file)

Below we started playing with h5py

But seemed more complex and not saving space ...


import h5py
import numpy as np


def save_warpmodel(group, model):

    # top-level metadata
    for key, value in model.items():

        if key == "mdict":
            continue

        group.attrs[key] = value

    mdict = model["mdict"]

    # lceval
    lceval_grp = group.create_group("lceval")

    for key, value in mdict["lceval"].items():
        lceval_grp.attrs[key] = value

    # corrmodel
    corr_grp = group.create_group("corrmodel")

    corrmodel = mdict["corrmodel"]

    for key in ["phase", "wave", "flux"]:

        arr = np.asarray(corrmodel[key])

        corr_grp.create_dataset(
            key,
            data=arr.astype(np.float32),
            compression="gzip",
            shuffle=True,
        )

with h5py.File("/Users/jnordin/tmp/warpmodels.h5", "w") as f:

    for snname, models in full_warplist_III.items():

        sn_grp = f.create_group(snname)

        for i, model in enumerate(models):

            model_grp = sn_grp.create_group(str(i))

            save_warpmodel(model_grp, model)

# read meatadata
with h5py.File("warpmodels.h5", "r") as f:

    attrs = dict(f["ZTF20aaerxne"]["0"].attrs)

print(attrs["peakmag"])

# read an object
with h5py.File("warpmodels.h5", "r") as f:

    flux = f["ZTF20aaerxne"]["0"]["corrmodel"]["flux"][:]

# loading many
with h5py.File("warpmodels.h5", "r") as f:

    for snname in f:

        for modelid in f[snname]:

            attrs = f[snname][modelid].attrs

            if attrs["peakmag"] < 18:
                print(snname, attrs["class"])